In [ ]:
# from huggingface_hub import snapshot_download
# snapshot_download(repo_id="Qwen/Qwen2.5-VL-3B-Instruct", local_dir="D:/projects/socialsense/huggingface_cache/qwen2.5-vl-3b", local_dir_use_symlinks=False)
# snapshot_download(repo_id="llava-hf/llava-onevision-qwen2-0.5b-ov-hf", local_dir="D:/projects/socialsense/huggingface_cache/llava-onevision-0.5b", local_dir_use_symlinks=False)
# snapshot_download(repo_id="deepseek-community/deepseek-vl-1.3b-chat", local_dir="D:/projects/socialsense/huggingface_cache/deepseek-vl-1.3b", local_dir_use_symlinks=False)
# try llava-onevision-qwen2-0.5b-si-hf

# legacy Qwen

## imports

In [ ]:
import pandas as pd
import csv
import os
import torch
from PIL import Image
from tqdm import tqdm
import datetime
from pathlib import Path

## setup model

In [ ]:
from transformers import LlavaOnevisionForConditionalGeneration
from transformers import Qwen2_5_VLForConditionalGeneration
from transformers import AutoProcessor, BitsAndBytesConfig
import torch

MODELNAME = "qwen2.5-vl-3b"
# MODELNAME = "llava-onevision-0.5b"
model_path = f"D:/projects/socialsense/huggingface_cache/{MODELNAME}"


# 4-bit quantization - fits easily in 6GB
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# model = LlavaOnevisionForConditionalGeneration.from_pretrained(
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    quantization_config=quant_config,
    device_map="auto",
    # offload_folder="offload",  # CPU offload dir
    # low_cpu_mem_usage=True,
    local_files_only=True
)

model = torch.compile(model, mode="reduce-overhead")
processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)

In [ ]:
# #Example usage
# from qwen_vl_utils import process_vision_info
# from pathlib import Path
# import torch
# from PIL import Image
# import re

# IMG_PATH = Path("D:/projects/socialsense/data/images_256x144/BigOffice-3_256.png").as_posix()
# image = Image.open(IMG_PATH)
# PROMPT = """
# On a scale from 1-5 (1-very inappropriate to 5-very appropriate), 
# how approrpaite would be for the robot to start a conversation? 
# Respond with only a single number - the appropriateness score.
# """

# messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT}]}]
    
# with torch.no_grad():
#     text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#     image_inputs, video_inputs = process_vision_info(messages)

#     inputs = qwen_processor(
#         text=[text], images=image_inputs, videos=video_inputs,
#         padding=True, return_tensors="pt"
#     ).to(qwen_model.device)

#     generated_ids = qwen_model.generate(
#         **inputs, 
#         max_new_tokens=1, 
#         do_sample=False,
#         pad_token_id=qwen_processor.tokenizer.eos_token_id,
#         # output_scores=True, 
#         # return_dict_in_generate=True,
#     )

#     output = qwen_processor.decode(
#         generated_ids[0][inputs['input_ids'].shape[1]:], 
#         skip_special_tokens=True
#     ).strip()

# print(f"Raw output: '{output}'")

## utils

In [ ]:
def create_prompt(action: str) -> str:
    """
    Image-based adaptation of a T5 text prompt for evaluating the predicted
    human-rated social appropriateness of robot actions.
    Original source: https://github.com/clear-nus/llm-human-model
    """
    normalized_action = (
        action.lower()
        .replace("ping", "")
        .replace("ing", "")
        .strip()
    )

    prompt = (
        "The image depicts a social scenario involving a robot. "
        "A human evaluator is shown this image and asked to rate how socially appropriate it is "
        f"for the robot to {normalized_action}. "
        "Answer choices: A. very inappropriate, B. inappropriate, C. neutral, D. appropriate, E. very appropriate. "
        "Question: Which answer will the human evaluator choose? "
        "Instruction: Respond only with a single letter (A–E) representing the choice."
    )

    return prompt

In [ ]:
def tokenize_answer_choices(answer_choices, processor):
    """Exact T5 equivalent - handles ANY tokenizer safely."""
    tokenized_candidate_answers = []
    for a in answer_choices:
        a_token = processor.tokenizer(a).input_ids
        a_token = a_token[0]  # Single token
        assert processor.tokenizer.decode(a_token) == a, f"Tokenization failed: {a}"
        tokenized_candidate_answers.append(a_token)
    return tokenized_candidate_answers

def get_probs(image, prompt, score_tokens, model, processor):
    """
    Single-sample equivalent of T5's get_probs_t5().
    Returns [P(A), P(B), P(C), P(D), P(E)] for one image+prompt.
    Original source: https://github.com/clear-nus/llm-human-model
    """
    
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image}, 
        {"type": "text", "text": prompt}
    ]}]
    
    with torch.no_grad():
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        
        inputs = processor(
            text=[text], images=[image],
            padding=True, return_tensors="pt"
        ).to(model.device)
        
        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=1, 
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
            output_scores=True, 
            return_dict_in_generate=True,
        )
        
        # Extract next-token probs
        scores = generated_ids.scores[0][0]  # [vocab_size]
        full_probs = torch.nn.functional.softmax(scores, dim=0)
        answer_probs = full_probs[torch.tensor(score_tokens).to(scores.device)].cpu().tolist()
    
    return answer_probs

## Evaluate the VLM on a test set

In [ ]:
def setup_results_file(results_csv_path, answer_choices, robot_actions_list):
    if not os.path.exists(results_csv_path):
        with open(results_csv_path, 'w', newline='', encoding='utf-8') as results_file:
            writer = csv.writer(results_file)
            header = ['image_path', 'prompt', 'action'] + answer_choices + ['sum of prob']
            writer.writerow(header)
        return 0,0
    else:
        with open(results_csv_path, 'r') as f:
            lines = sum(1 for _ in f)
        
        total_rows = lines - 1
        start_img_idx = total_rows // len(robot_actions_list)
        start_action_idx = total_rows % len(robot_actions_list)
        return start_img_idx, start_action_idx

In [ ]:
def run_vlm(image_paths, answer_choices, robot_actions_list, model, processor, results_csv_path, start_img_idx, start_action_idx):
    with open(results_csv_path, 'a', newline='', encoding='utf-8') as results_file:
        writer = csv.writer(results_file)
        
        score_tokens = tokenize_answer_choices(answer_choices, processor)
        
        pbar = tqdm(range(start_img_idx, len(image_paths)), desc="Processing images")
        for img_idx in pbar:
            img_path = image_paths[img_idx]
            
            # Skip actions if resuming mid-image
            action_start = start_action_idx if img_idx == start_img_idx else 0
            
            try:
                image = Image.open(img_path).convert('RGB')#.resize((254, 144), Image.BILINEAR)
            except Exception as e:
                print(f"❌ Failed to load {img_path}: {e}")
                continue
            
            for action_idx in range(action_start, len(robot_actions_list)):
                action = robot_actions_list[action_idx]
                
                prompt = create_prompt(action)
                try:
                    answer_probs = get_probs(image, prompt, score_tokens, model, processor)
                    sum_of_probs = sum(answer_probs)
                    
                    result_row = [img_path, prompt, action] + answer_probs + [sum_of_probs]
                    writer.writerow(result_row)
                    results_file.flush()  # Ensure written immediately
                    
                    pbar.set_description(f"Img {img_idx}/{len(image_paths)} Act {action_idx+1}/{len(robot_actions_list)}")
                    
                except Exception as e:
                    print(f"❌ Error processing {img_path} + {action}: {e}")
                    # Write dummy row to maintain alignment
                    dummy_probs = ["ERROR"] * 5
                    dummy_row = [img_path, prompt, action] + dummy_probs + ["ERROR"]
                    writer.writerow(dummy_row)
                    results_file.flush()
                    continue

            
            # Reset for next image
            if img_idx == start_img_idx:
                start_action_idx = 0

    print(f"✅ COMPLETE! Results saved to {results_csv_path}")


In [ ]:
df = pd.read_pickle("../data/mean_data_pepper_test.pkl")
image_paths = df['image_path'].tolist()
# Let VLM use pre-downsized images for efficiency
image_paths = [p.replace("data/images/", "data/images_256x144/") for p in image_paths]

answer_choices = ['A', 'B', 'C', 'D', 'E']
robot_actions_list = [
    'Vaccum Cleaning', 'Mopping the Floor', 'Carry Warm Food', 'Carry Cold Food',
    'Carry Drinks', 'Carry Small Objects', 'Carry Large Objects', 'Cleaning',
    'Starting a conversation'
]
results_csv_path = f"./results/{MODELNAME}_test_results1.csv"

start_img_idx, start_action_idx = setup_results_file(results_csv_path, answer_choices, robot_actions_list)
if start_img_idx != 0:
    print(f"Resume from image {start_img_idx}/{len(image_paths)}, action {start_action_idx}/{len(robot_actions_list)}")

In [ ]:
run_vlm(image_paths, answer_choices, robot_actions_list, model, processor, results_csv_path, start_img_idx, start_action_idx)

## 5 fold inference

In [ ]:
from sklearn.model_selection import StratifiedKFold

answer_choices = ['A', 'B', 'C', 'D', 'E']
robot_actions_list = [
    'Vaccum Cleaning', 'Mopping the Floor', 'Carry Warm Food', 'Carry Cold Food',
    'Carry Drinks', 'Carry Small Objects', 'Carry Large Objects', 'Cleaning',
    'Starting a conversation'
]

df = pd.read_pickle('../data/mean_data_pepper.pkl')


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_loaders = {}

# Loop over each fold
for fold_num, (train_val_idx, test_idx) in enumerate(skf.split(df, df['domain'])):
    # if fold_num == 0:
    #     continue
    # train_val_df = df.iloc[train_val_idx]
    test_df = df.iloc[test_idx]
    image_paths = test_df['image_path'].tolist()
    image_paths = [p.replace("data/images/", "data/images_256x144/") for p in image_paths]
    results_csv_path = f"./results/{MODELNAME}_test_results_fold{fold_num}.csv"
    start_img_idx, start_action_idx = setup_results_file(results_csv_path, answer_choices, robot_actions_list)
    assert start_img_idx == 0
    run_vlm(image_paths, answer_choices, robot_actions_list, model, processor, results_csv_path, start_img_idx, start_action_idx)


## analyse results

In [ ]:
for i in range(5):    
    #test results csv for errors
    results_csv_path = f"./results/{MODELNAME}_test_results_fold{fold_num}.csv"
    results_df = pd.read_csv(results_csv_path)
    total_rows = len(results_df)

    # Count dummies: ANY "ERROR" in A-E columns OR sum_of_prob == "ERROR"
    is_dummy = (
        results_df[answer_choices].apply(lambda x: x.astype(str).str.contains('ERROR', na=False).any(), axis=1) |
        (results_df['sum of prob'].astype(str) == 'ERROR')
    ).sum()

    print(f"❌ Dummy rows: {is_dummy}")
    print(f"✅ Success rate: {100*(1 - is_dummy/total_rows):.1f}%")

In [ ]:
def vlm_csv_to_domain_predictions(vlm_preds_df, targets_df):
    """
    Convert VLM CSV to exact infer_per_domain structure.
    1.Computes weighted average from vlm token probability distribution for each score token ("A", "B", "C", "D", "E").
    2.Normalises the score from 1-5 to 0-1.
    3.Creates a dictionary with domain grouped numpy arrays. Each array has weighted score for each action (columns) and each image (rows) in the test dataset.
    """
    import pandas as pd
    import numpy as np
    import torch
    
    token_probability_cols = ['A', 'B', 'C', 'D', 'E']
    
    # Weighted scores
    probs_df = vlm_preds_df[token_probability_cols].astype(float)
    likert_weights = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    vlm_preds_df['weighted_score'] = np.dot(probs_df, likert_weights)
    
    action_cols = ['Vaccum Cleaning', 'Mopping the Floor', 'Carry Warm Food', 
                   'Carry Cold Food', 'Carry Drinks', 'Carry Small Objects', 
                   'Carry Large Objects', 'Cleaning', 'Starting a conversation']
    
    preds_pivot_df = vlm_preds_df.pivot(index='image_path', columns='action', values='weighted_score')[action_cols]
    
    # Reorder rows to match test set
    targets_df = targets_df[['image_path', 'domain']+action_cols].set_index('image_path')
    targets_preds_df = targets_df.join(preds_pivot_df, on='image_path', how='inner', rsuffix='_pred')

    pred_cols = [f"{a}_pred" for a in action_cols]

    #Ensure join was correct, no values were lost
    assert not targets_preds_df.isna().any().any()
    assert len(preds_pivot_df) == len(targets_preds_df)
    
    # Get numpy matrix with predictions and targets
    preds = targets_preds_df[pred_cols].values.astype(np.float32)
    targets = targets_preds_df[action_cols].values.astype(np.float32)

    # Normalise
    preds = (preds-1)/4
    targets = (targets-1)/4

    domain_data = {}
    for domain in targets_df['domain'].unique():
        domain_mask = targets_preds_df['domain'] == domain
        
        domain_preds = preds[domain_mask]
        domain_targets = targets[domain_mask]
        
        domain_data[domain] = (
            torch.from_numpy(domain_preds),
            torch.from_numpy(domain_targets)
        )
        
    return domain_data

### single file

In [ ]:
#test results csv for errors
results_csv_path = f"./results/{MODELNAME}_test_results.csv"
results_df = pd.read_csv(results_csv_path)
total_rows = len(results_df)

# Count dummies: ANY "ERROR" in A-E columns OR sum_of_prob == "ERROR"
is_dummy = (
    results_df[answer_choices].apply(lambda x: x.astype(str).str.contains('ERROR', na=False).any(), axis=1) |
    (results_df['sum of prob'].astype(str) == 'ERROR')
).sum()

print(f"❌ Dummy rows: {is_dummy}")
print(f"✅ Success rate: {100*(1 - is_dummy/total_rows):.1f}%")

In [ ]:
vlm_df = pd.read_csv(f'./results/{MODELNAME}_test_results.csv')
vlm_df["image_path"] = vlm_df["image_path"].str.replace("data/images_256x144/", "data/images/", regex=False)

labels_df = pd.read_pickle("../data/mean_data_pepper_test.pkl")

domain_test_preds_targets = vlm_csv_to_domain_predictions(vlm_df, labels_df)

In [ ]:
# Save the results in a pickle dict
# import pickle
# with open('./results/vlm_qwen25_domain_preds_targets.pkl', 'wb') as f:
#     model_results = {"qwen2.5-vl-3b": {"domain_test_preds_targets": domain_test_preds_targets}}
#     pickle.dump(model_results, f)

### 5 folds evaluation

In [ ]:
model_results = {}
for fold_num in range(5):
    results_csv_path = f"./results/{MODELNAME}_test_results_fold{fold_num}.csv"
    vlm_df = pd.read_csv(results_csv_path)
    vlm_df["image_path"] = vlm_df["image_path"].str.replace("data/images_256x144/", "data/images/", regex=False)

    labels_df = pd.read_pickle("../data/mean_data_pepper.pkl")

    domain_test_preds_targets = vlm_csv_to_domain_predictions(vlm_df, labels_df)

    model_results[f'{MODELNAME}_fold={fold_num}'] = {"domain_test_preds_targets": domain_test_preds_targets}

# import pickle
# with open('./results/vlm_{MODELNAME}_domain_preds_targets_folds.pkl', 'wb') as f:
#     pickle.dump(model_results, f)

# Qwen/LLaVA/Deepseek

## single script

In [ ]:
import pandas as pd
import csv
import os
import torch
from PIL import Image
from tqdm import tqdm
import datetime
from pathlib import Path
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import AutoModelForImageTextToText

for MODELNAME in ["deepseek-vl-1.3b", "llava-onevision-0.5b"]:
# MODELNAME = "deepseek-vl-1.3b"
# MODELNAME = "qwen2.5-vl-3b"
# MODELNAME = "llava-onevision-0.5b"
    model_path = f"D:/projects/socialsense/huggingface_cache/{MODELNAME}"


    # # 4-bit quantization - fits easily in 6GB
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForImageTextToText.from_pretrained(
        model_path,
        quantization_config=quant_config,
        # torch_dtype=torch.float16,
        device_map="auto",
        # offload_folder="offload",  # CPU offload dir
        # low_cpu_mem_usage=True,
        local_files_only=True
    )

    model = torch.compile(model, mode="reduce-overhead")
    processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)


    from sklearn.model_selection import StratifiedKFold

    answer_choices = ['A', 'B', 'C', 'D', 'E']
    robot_actions_list = [
        'Vaccum Cleaning', 'Mopping the Floor', 'Carry Warm Food', 'Carry Cold Food',
        'Carry Drinks', 'Carry Small Objects', 'Carry Large Objects', 'Cleaning',
        'Starting a conversation'
    ]
    df = pd.read_pickle('../data/mean_data_pepper.pkl')

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold_loaders = {}
    for fold_num, (train_val_idx, test_idx) in enumerate(skf.split(df, df['domain'])):
        test_df = df.iloc[test_idx]
        image_paths = test_df['image_path'].tolist()
        image_paths = [p.replace("data/images/", "data/images_256x144/") for p in image_paths]
        results_csv_path = f"./results/{MODELNAME}_test_results_fold{fold_num}.csv"
        start_img_idx, start_action_idx = setup_results_file(results_csv_path, answer_choices, robot_actions_list)
        assert start_img_idx == 0
        run_vlm(image_paths, answer_choices, robot_actions_list, model, processor, results_csv_path, start_img_idx, start_action_idx)


## imports

In [ ]:
import pandas as pd
import csv
import os
import torch
from PIL import Image
from tqdm import tqdm
import datetime
from pathlib import Path

## setup model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, LlavaOnevisionForConditionalGeneration, DeepseekVLForConditionalGeneration
from transformers import AutoProcessor, BitsAndBytesConfig
from transformers import AutoModelForImageTextToText

MODELNAME = "deepseek-vl-1.3b"
# MODELNAME = "qwen2.5-vl-3b"
# MODELNAME = "llava-onevision-0.5b"
model_path = f"D:/projects/socialsense/huggingface_cache/{MODELNAME}"


# # 4-bit quantization - fits easily in 6GB
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    quantization_config=quant_config,
    # torch_dtype=torch.float16,
    device_map="auto",
    # offload_folder="offload",  # CPU offload dir
    # low_cpu_mem_usage=True,
    local_files_only=True
)

model = torch.compile(model, mode="reduce-overhead")
processor = AutoProcessor.from_pretrained(model_path, local_files_only=True)

print(model.dtype)
print(next(model.parameters()).dtype)


In [ ]:
# #Example usage
# from pathlib import Path
# import torch
# from PIL import Image
# import re

# IMG_PATH = Path("D:/projects/socialsense/data/images_256x144/BigOffice-3_256.png").as_posix()
# image = Image.open(IMG_PATH)
# PROMPT = """
# On a scale from 1-5 (1-very inappropriate to 5-very appropriate), 
# how approrpaite would be for the robot to start a conversation? 
# Respond with only a single number - the appropriateness score.
# """
# score_tokens = tokenize_answer_choices(['1', '2', '3', '4', '5'], processor)

# messages = [{"role": "user", "content": [{"type": "image", "image": image},{"type": "text", "text": PROMPT}]}]

# # text = processor.apply_chat_template(
# #     messages, 
# #     add_generation_prompt=True,
# #     tokenize=False
# #     )
# # inputs = processor(
# #     text=text, 
# #     images=image,
# #     # padding=True, 
# #     return_tensors="pt"
# #     ).to(model.device, dtype=model.dtype)

# inputs = processor.apply_chat_template(
#     messages,
#     add_generation_prompt=True,
#     tokenize=True,
#     return_tensors="pt",
#     return_dict=True,
#     ).to(model.device, dtype=model.dtype)

# with torch.no_grad():
#     generated_ids = model.generate(
#         **inputs, 
#         max_new_tokens=1, 
#         do_sample=False,
#         pad_token_id=processor.tokenizer.eos_token_id,
#         output_scores=True, 
#         return_dict_in_generate=True,
#         )

#     # Next-token logits -> probs
#     scores = generated_ids.scores[0][0]  # [vocab_size]
#     full_probs = torch.nn.functional.softmax(scores.float(), dim=0)

#     # Extract probs for A–E
#     idx = torch.tensor(score_tokens, device=scores.device)
#     answer_probs = full_probs[idx].cpu().tolist()
    
# print(answer_probs)
# print(sum(answer_probs))


## utils

In [ ]:
def create_prompt(action: str) -> str:
    """
    Image-based adaptation of a T5 text prompt for evaluating the predicted
    human-rated social appropriateness of robot actions.
    Original source: https://github.com/clear-nus/llm-human-model
    """
    normalized_action = (
        action.lower()
        .replace("ping", "")
        .replace("ing", "")
        .strip()
    )

    prompt = (
        "The image depicts a social scenario involving a robot. "
        "A human evaluator is shown this image and asked to rate how socially appropriate it is "
        f"for the robot to {normalized_action}. "
        "Answer choices: A. very inappropriate, B. inappropriate, C. neutral, D. appropriate, E. very appropriate. "
        "Question: Which answer will the human evaluator choose? "
        "Instruction: Respond only with a single letter (A–E) representing the choice."
    )

    return prompt

In [ ]:
def tokenize_answer_choices(answer_choices, processor):
    """Exact T5 equivalent - handles ANY tokenizer safely."""
    tokenized_candidate_answers = []
    for a in answer_choices:
        ids = processor.tokenizer(a, add_special_tokens=False).input_ids
        assert len(ids) == 1, f"{a!r} is not a single token: {ids}"
        a_token = ids[0]
        decoded = processor.tokenizer.decode([a_token])
        assert decoded.strip() == a, f"Tokenization failed for {a!r}: got {decoded!r}"
        tokenized_candidate_answers.append(a_token)
    return tokenized_candidate_answers

In [ ]:
def get_probs(image, prompt, score_tokens, model, processor):
    """
    Returns [P(A), P(B), P(C), P(D), P(E)] for one image+text.
    """
    with torch.no_grad():
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
            ]}]
        
        # text = processor.apply_chat_template(
        #     messages, 
        #     add_generation_prompt=True,
        #     tokenize=False
        #     )
        # inputs = processor(
        #     text=text, 
        #     images=image,
        #     padding=True, 
        #     return_tensors="pt"
        #     ).to(model.device, dtype=model.dtype)

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
            ).to(model.device, dtype=model.dtype)

        generated_ids = model.generate(
            **inputs, 
            max_new_tokens=1, 
            do_sample=False,
            pad_token_id=processor.tokenizer.eos_token_id,
            output_scores=True, 
            return_dict_in_generate=True,
            )

        # Next-token logits -> probs
        scores = generated_ids.scores[0][0]  # [vocab_size]
        full_probs = torch.nn.functional.softmax(scores.float(), dim=0)

        # Extract probs for A–E
        idx = torch.tensor(score_tokens, device=scores.device)
        answer_probs = full_probs[idx].cpu().tolist()
        
    return answer_probs


In [ ]:
def setup_results_file(results_csv_path, answer_choices, robot_actions_list):
    if not os.path.exists(results_csv_path):
        with open(results_csv_path, 'w', newline='', encoding='utf-8') as results_file:
            writer = csv.writer(results_file)
            header = ['image_path', 'prompt', 'action'] + answer_choices + ['sum of prob']
            writer.writerow(header)
        return 0,0
    else:
        with open(results_csv_path, 'r') as f:
            lines = sum(1 for _ in f)
        
        total_rows = lines - 1
        start_img_idx = total_rows // len(robot_actions_list)
        start_action_idx = total_rows % len(robot_actions_list)
        return start_img_idx, start_action_idx

In [ ]:
def run_vlm(image_paths, answer_choices, robot_actions_list, model, processor, results_csv_path, start_img_idx, start_action_idx):
    with open(results_csv_path, 'a', newline='', encoding='utf-8') as results_file:
        writer = csv.writer(results_file)
        
        score_tokens = tokenize_answer_choices(answer_choices, processor)
        
        pbar = tqdm(range(start_img_idx, len(image_paths)), desc="Processing images")
        for img_idx in pbar:
            img_path = image_paths[img_idx]
            
            # Skip actions if resuming mid-image
            action_start = start_action_idx if img_idx == start_img_idx else 0
            
            try:
                image = Image.open(img_path).convert('RGB')#.resize((254, 144), Image.BILINEAR)
            except Exception as e:
                print(f"❌ Failed to load {img_path}: {e}")
                continue
            
            for action_idx in range(action_start, len(robot_actions_list)):
                action = robot_actions_list[action_idx]
                
                prompt = create_prompt(action)
                try:
                    answer_probs = get_probs(image, prompt, score_tokens, model, processor)
                    sum_of_probs = sum(answer_probs)
                    
                    result_row = [img_path, prompt, action] + answer_probs + [sum_of_probs]
                    writer.writerow(result_row)
                    results_file.flush()  # Ensure written immediately
                    
                    pbar.set_description(f"Img {img_idx}/{len(image_paths)} Act {action_idx+1}/{len(robot_actions_list)}")
                    
                except Exception as e:
                    print(f"❌ Error processing {img_path} + {action}: {e}")
                    # Write dummy row to maintain alignment
                    dummy_probs = ["ERROR"] * 5
                    dummy_row = [img_path, prompt, action] + dummy_probs + ["ERROR"]
                    writer.writerow(dummy_row)
                    results_file.flush()
                    continue

            
            # Reset for next image
            if img_idx == start_img_idx:
                start_action_idx = 0

    print(f"✅ COMPLETE! Results saved to {results_csv_path}")


## Evaluate the VLM on a test set

In [ ]:
df = pd.read_pickle("../data/mean_data_pepper_test.pkl")
image_paths = df['image_path'].tolist()
# Let VLM use pre-downsized images for efficiency
image_paths = [p.replace("data/images/", "data/images_256x144/") for p in image_paths]

answer_choices = ['A', 'B', 'C', 'D', 'E']
robot_actions_list = [
    'Vaccum Cleaning', 'Mopping the Floor', 'Carry Warm Food', 'Carry Cold Food',
    'Carry Drinks', 'Carry Small Objects', 'Carry Large Objects', 'Cleaning',
    'Starting a conversation'
]
results_csv_path = f"./results/{MODELNAME}_test_results.csv"

start_img_idx, start_action_idx = setup_results_file(results_csv_path, answer_choices, robot_actions_list)
if start_img_idx != 0:
    print(f"Resume from image {start_img_idx}/{len(image_paths)}, action {start_action_idx}/{len(robot_actions_list)}")

In [ ]:
run_vlm(image_paths, answer_choices, robot_actions_list, model, processor, results_csv_path, start_img_idx, start_action_idx)

## 5 fold inference

In [ ]:
from sklearn.model_selection import StratifiedKFold

answer_choices = ['A', 'B', 'C', 'D', 'E']
robot_actions_list = [
    'Vaccum Cleaning', 'Mopping the Floor', 'Carry Warm Food', 'Carry Cold Food',
    'Carry Drinks', 'Carry Small Objects', 'Carry Large Objects', 'Cleaning',
    'Starting a conversation'
]

df = pd.read_pickle('../data/mean_data_pepper.pkl')


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_loaders = {}

# Loop over each fold
for fold_num, (train_val_idx, test_idx) in enumerate(skf.split(df, df['domain'])):
    # if fold_num == 0:
    #     continue
    # train_val_df = df.iloc[train_val_idx]
    test_df = df.iloc[test_idx]
    image_paths = test_df['image_path'].tolist()
    image_paths = [p.replace("data/images/", "data/images_256x144/") for p in image_paths]
    results_csv_path = f"./results/{MODELNAME}_test_results_fold{fold_num}.csv"
    start_img_idx, start_action_idx = setup_results_file(results_csv_path, answer_choices, robot_actions_list)
    assert start_img_idx == 0
    run_vlm(image_paths, answer_choices, robot_actions_list, model, processor, results_csv_path, start_img_idx, start_action_idx)


## analyse results

In [ ]:
MODELNAME = "deepseek-vl-1.3b"#, 
# MODELNAME = "llava-onevision-0.5b"

In [ ]:
for i in range(5):    
    #test results csv for errors
    results_csv_path = f"./results/{MODELNAME}_test_results_fold{fold_num}.csv"
    results_df = pd.read_csv(results_csv_path)
    total_rows = len(results_df)

    # Count dummies: ANY "ERROR" in A-E columns OR sum_of_prob == "ERROR"
    is_dummy = (
        results_df[answer_choices].apply(lambda x: x.astype(str).str.contains('ERROR', na=False).any(), axis=1) |
        (results_df['sum of prob'].astype(str) == 'ERROR')
    ).sum()

    print(f"❌ Dummy rows: {is_dummy}")
    print(f"✅ Success rate: {100*(1 - is_dummy/total_rows):.1f}%")

In [ ]:
def vlm_csv_to_domain_predictions(vlm_preds_df, targets_df):
    """
    Convert VLM CSV to exact infer_per_domain structure.
    1.Computes weighted average from vlm token probability distribution for each score token ("A", "B", "C", "D", "E").
    2.Normalises the score from 1-5 to 0-1.
    3.Creates a dictionary with domain grouped numpy arrays. Each array has weighted score for each action (columns) and each image (rows) in the test dataset.
    """
    import pandas as pd
    import numpy as np
    import torch
    
    token_probability_cols = ['A', 'B', 'C', 'D', 'E']
    
    # Weighted scores
    probs_df = vlm_preds_df[token_probability_cols].astype(float)
    likert_weights = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
    vlm_preds_df['weighted_score'] = np.dot(probs_df, likert_weights)
    
    action_cols = ['Vaccum Cleaning', 'Mopping the Floor', 'Carry Warm Food', 
                   'Carry Cold Food', 'Carry Drinks', 'Carry Small Objects', 
                   'Carry Large Objects', 'Cleaning', 'Starting a conversation']
    
    preds_pivot_df = vlm_preds_df.pivot(index='image_path', columns='action', values='weighted_score')[action_cols]
    
    # Reorder rows to match test set
    targets_df = targets_df[['image_path', 'domain']+action_cols].set_index('image_path')
    targets_preds_df = targets_df.join(preds_pivot_df, on='image_path', how='inner', rsuffix='_pred')

    pred_cols = [f"{a}_pred" for a in action_cols]

    #Ensure join was correct, no values were lost
    assert not targets_preds_df.isna().any().any()
    assert len(preds_pivot_df) == len(targets_preds_df)
    
    # Get numpy matrix with predictions and targets
    preds = targets_preds_df[pred_cols].values.astype(np.float32)
    targets = targets_preds_df[action_cols].values.astype(np.float32)

    # Normalise
    preds = (preds-1)/4
    targets = (targets-1)/4

    domain_data = {}
    for domain in targets_df['domain'].unique():
        domain_mask = targets_preds_df['domain'] == domain
        
        domain_preds = preds[domain_mask]
        domain_targets = targets[domain_mask]
        
        domain_data[domain] = (
            torch.from_numpy(domain_preds),
            torch.from_numpy(domain_targets)
        )
        
    return domain_data

### single file

In [ ]:
#test results csv for errors
results_csv_path = f"./results/{MODELNAME}_test_results.csv"
results_df = pd.read_csv(results_csv_path)
total_rows = len(results_df)

# Count dummies: ANY "ERROR" in A-E columns OR sum_of_prob == "ERROR"
is_dummy = (
    results_df[answer_choices].apply(lambda x: x.astype(str).str.contains('ERROR', na=False).any(), axis=1) |
    (results_df['sum of prob'].astype(str) == 'ERROR')
).sum()

print(f"❌ Dummy rows: {is_dummy}")
print(f"✅ Success rate: {100*(1 - is_dummy/total_rows):.1f}%")

In [ ]:
vlm_df = pd.read_csv(f'./results/{MODELNAME}_test_results.csv')
vlm_df["image_path"] = vlm_df["image_path"].str.replace("data/images_256x144/", "data/images/", regex=False)

labels_df = pd.read_pickle("../data/mean_data_pepper_test.pkl")

domain_test_preds_targets = vlm_csv_to_domain_predictions(vlm_df, labels_df)

In [ ]:
# Save the results in a pickle dict
# import pickle
# with open('./results/vlm_qwen25_domain_preds_targets.pkl', 'wb') as f:
#     model_results = {"qwen2.5-vl-3b": {"domain_test_preds_targets": domain_test_preds_targets}}
#     pickle.dump(model_results, f)

### 5 folds evaluation

In [ ]:
# MODELNAME = "deepseek-vl-1.3b"
# MODELNAME = "qwen2.5-vl-3b"
MODELNAME = "llava-onevision-0.5b"

In [ ]:
model_results = {}
for fold_num in range(5):
    results_csv_path = f"./results/{MODELNAME}_test_results_fold{fold_num}.csv"
    vlm_df = pd.read_csv(results_csv_path)
    vlm_df["image_path"] = vlm_df["image_path"].str.replace("data/images_256x144/", "data/images/", regex=False)

    labels_df = pd.read_pickle("../data/mean_data_pepper.pkl")

    domain_test_preds_targets = vlm_csv_to_domain_predictions(vlm_df, labels_df)

    model_results[f'{MODELNAME}_fold={fold_num}'] = {"domain_test_preds_targets": domain_test_preds_targets}

import pickle
with open(f'./results/vlm_{MODELNAME}_domain_preds_targets_folds.pkl', 'wb') as f:
    pickle.dump(model_results, f)